<a href="https://colab.research.google.com/github/Nityakothavari7/EmberMind/blob/main/06_qwen_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers faiss-cpu transformers peft

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.2 MB/s eta 0:00:00


In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!pip show torchao

Name: torchao
Version: 0.10.0
Summary: Package for applying ao techniques to GPU models
Home-page: https://github.com/pytorch/ao
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 


In [ ]:
import peft
import transformers

print(peft.__version__)
print(transformers.__version__)

0.19.1
5.10.2


In [ ]:
import json
import faiss
import pickle
import numpy as np
import torch

from sentence_transformers import SentenceTransformer

print("Libraries Loaded")

Libraries Loaded


In [ ]:
MEMORY_FILE = "/content/drive/MyDrive/user_memory.json"

with open(MEMORY_FILE, "r") as f:
    memories = json.load(f)

print("Memories:", len(memories))

Memories: 6


In [ ]:
memory_index = faiss.read_index(
    "/content/drive/MyDrive/memory_index.faiss"
)

print(
    "Memory Index:",
    memory_index.ntotal
)

Memory Index: 6


In [ ]:
therapy_index = faiss.read_index(
    "/content/drive/MyDrive/therapy_index.faiss"
)

print(
    "Therapy Index:",
    therapy_index.ntotal
)

Therapy Index: 16994


In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/ember_qwen_lora"
)

model.eval()

print("EMBER-Qwen Loaded Successfully")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

EMBER-Qwen Loaded Successfully


In [ ]:
CRISIS_TERMS = [

    "suicide",
    "suicidal",

    "kill myself",

    "end my life",

    "want to die",

    "don't want to live",

    "self harm",

    "hurt myself",

    "jump",

    "jump off",

    "jump off building",

    "standing on edge",

    "terrace",

    "overdose"
]

In [ ]:
def detect_risk(text):

    text = text.lower()

    for term in CRISIS_TERMS:

        if term in text:

            return "high"

    return "low"

In [ ]:
CRISIS_MESSAGE = """

If you are in immediate danger or feel you may act on thoughts of self-harm:

India:
Tele-MANAS: 14416
or 1-800-891-4416

Please contact:
• A trusted family member
• A close friend
• A mental health professional
• Emergency services if needed

You do not have to face this alone.
"""

In [ ]:
print(
    detect_risk(
        "I want to kill myself"
    )
)

print(
    detect_risk(
        "I am stressed about exams"
    )
)

high
low


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.1 MB/s eta 0:00:00


In [ ]:
import json
import faiss
import numpy as np
from datetime import datetime

class MemoryManager:

    def __init__(
        self,
        memory_file,
        index_file,
        embedding_model
    ):

        self.memory_file = memory_file
        self.index_file = index_file
        self.embedding_model = embedding_model

        self.load()
    def load(self):

        try:

            with open(
                self.memory_file,
                "r"
            ) as f:

                self.memories = json.load(f)

        except:

            self.memories = []

        if len(self.memories) > 0:

            vectors = np.array(
                [
                    m["embedding"]
                    for m in self.memories
                ],
                dtype=np.float32
            )

            faiss.normalize_L2(vectors)

            self.index = faiss.IndexFlatIP(
                vectors.shape[1]
            )

            self.index.add(vectors)

        else:

            self.index = None
    def save(self):

        with open(
            self.memory_file,
            "w"
        ) as f:

            json.dump(
                self.memories,
                f,
                indent=4
            )
    def calculate_importance(
        self,
        text
    ):

        keywords = [

            "anxiety",
            "depression",

            "nightmare",
            "stress",

            "lonely",
            "hopeless",

            "breakup",
            "trauma",

            "parents",
            "family",
            "comparison",
            "compare",

            "relationships",
            "rejection",

            "suicide",
            "kill myself"
        ]

        score = 0.1

        text = text.lower()

        for word in keywords:

            if word in text:

                score += 0.2

        return min(score, 1.0)
    def add_memory(
        self,
        user_message
    ):

        importance = self.calculate_importance(
            user_message
        )

        if importance < 0.3:

            return False

        embedding = self.embedding_model.encode(
            user_message
        ).tolist()

        memory = {

            "timestamp":
            str(datetime.now()),

            "user_message":
            user_message,

            "memory_summary":
            user_message[:120],

            "importance_score":
            importance,

            "embedding":
            embedding
        }

        self.memories.append(
            memory
        )

        self.save()

        self.rebuild_index()

        return True
    def rebuild_index(self):

        vectors = np.array(
            [
                m["embedding"]
                for m in self.memories
            ],
            dtype=np.float32
        )

        faiss.normalize_L2(
            vectors
        )

        self.index = faiss.IndexFlatIP(
            vectors.shape[1]
        )

        self.index.add(
            vectors
        )

        faiss.write_index(
            self.index,
            self.index_file
        )
    def retrieve(
        self,
        query,
        top_k=3
    ):

        if len(self.memories) == 0:

            return []

        query_vector = self.embedding_model.encode(
            [query]
        )

        query_vector = np.array(
            query_vector,
            dtype=np.float32
        )

        faiss.normalize_L2(
            query_vector
        )

        actual_k = min(
            top_k,
            len(self.memories)
        )

        scores, indices = self.index.search(
            query_vector,
            actual_k
        )

        return [
            self.memories[i]
            for i in indices[0]
        ]

In [ ]:
memory_manager = MemoryManager(

    memory_file=
    "/content/drive/MyDrive/user_memory.json",

    index_file=
    "/content/drive/MyDrive/memory_index.faiss",

    embedding_model=
    embedding_model
)

print(
    "Memories:",
    len(memory_manager.memories)
)

Memories: 6


In [ ]:
result = memory_manager.add_memory(
    "My parents compare me with others."
)

print(result)

True


In [ ]:
print(
    memory_manager.calculate_importance(
        "My parents compare me with others."
    )
)

0.5


In [ ]:
print(len(memory_manager.memories))

6


In [ ]:
import json

with open(memory_manager.memory_file, "r") as f:
    memories = json.load(f)

print(len(memories))
print(memories[-1]["user_message"])

6
My parents compare me with others.


In [ ]:
results = memory_manager.retrieve(
    "I feel stressed again",
    top_k=3
)

print(results)

[{'timestamp': '2026-06-15 18:37:51.526169', 'user_message': 'I feel stressed again.', 'memory_summary': 'I feel stressed again.', 'importance_score': 0.30000000000000004, 'embedding': [0.03886417672038078, -0.0886155515909195, 0.02594158425927162, 0.07714194059371948, 0.07262279838323593, -0.05697520449757576, 0.05157800763845444, -0.007186424918472767, -0.000868986186105758, -0.10089311003684998, -0.09472530335187912, 0.007810760755091906, -0.0537334643304348, 0.0320390984416008, 0.05693157762289047, -0.0022255422081798315, 0.041285790503025055, -0.047233711928129196, -0.04397137463092804, -0.016763782128691673, -0.009754421189427376, 0.015224379487335682, -0.05771269276738167, 0.06401879340410233, -0.021342337131500244, 0.09165043383836746, -0.0480448342859745, 0.034686461091041565, 0.013859489001333714, -0.07124470174312592, 0.02543003298342228, -0.006394898518919945, -0.0472203828394413, -0.007898660376667976, 0.057487111538648605, 0.03129206970334053, -0.047872383147478104, -0.00

In [ ]:
context = build_context(
    "I feel stressed again"
)

print(context)


[USER MEMORY]

- I feel stressed again.
- I am stressed because of exams.
- I am stressed about exams.

[THERAPY EXAMPLES]

Example 1:

User:
The level of stress in my life has become overwhelming, and I'm beginning to feel burnt out. My primary goal in counseling is to discover effective ways of managing stress and finding a better work-life balance.

Every day feels like a race against time, with never-ending deadlines and responsibilities piling up. It's difficult for me to unwind or relax because there's always something else demanding my attention. The constant pressure has started taking a toll on my physical and mental well-being.

One specific situation that recently triggered my stress levels was a major project at work. The amount of work required, combined with tight deadlines, pushed me to my limits. Even after completing the project, I couldn't fully relax because the next one was already looming over me.

The symptoms of stress are present almost constantly. I experience

In [ ]:
print("memory_manager" in globals())
print("retrieve_therapy_examples" in globals())
print("build_context" in globals())
print("detect_risk" in globals())
print("model" in globals())
print("tokenizer" in globals())

True
True
True
True
True
True


In [ ]:
import pickle

with open(
    "/content/drive/MyDrive/therapy_metadata.pkl",
    "rb"
) as f:

    therapy_docs = pickle.load(f)

print("Therapy Docs:", len(therapy_docs))

therapy_index = faiss.read_index(
    "/content/drive/MyDrive/therapy_index.faiss"
)

Therapy Docs: 16994


In [ ]:
## DEPENDENCY BLOCK

def retrieve_therapy_examples(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    scores, indices = therapy_index.search(
        query_embedding.astype(np.float32),
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({

            "score": float(score),

            "document": therapy_docs[idx]
        })

    return results
def build_context(
    user_message,
    therapy_k=3
):

    memories = memory_manager.retrieve(
    user_message,
    top_k=3
)

    therapies = retrieve_therapy_examples(
        user_message,
        therapy_k
    )

    context = ""

    context += "\n[USER MEMORY]\n\n"

    for item in memories:

        context += (
            f"- {item['memory_summary']}\n"
        )

    context += "\n[THERAPY EXAMPLES]\n\n"

    for idx, item in enumerate(
        therapies,
        start=1
    ):

        example = item["document"][:1000]

        context += (
            f"Example {idx}:\n"
        )

        context += example

        context += "\n\n"

    context += (
        "\n[CURRENT USER]\n\n"
    )

    context += user_message

    return context

In [ ]:
context = build_context(
    "I feel lonely"
)

print(context[:1000])


[USER MEMORY]

- I feel lonely and hopeless.

[THERAPY EXAMPLES]

Example 1:

User:
I've been feeling really lonely lately, even though I have friends and acquaintances around me. It's like there's a constant void that I can't fill. I crave deeper connections and meaningful relationships, but I struggle to open up and trust others. It feels like I'm always on the outside looking in, and it's starting to take a toll on my mental health. I want to learn how to build healthier relationships and overcome this sense of loneliness.

Therapist:
I can understand how difficult it must be to feel lonely even when you have people around you. It sounds like you're longing for more meaningful connections and struggling with trust issues. Building healthier relationships takes time and effort, but there are steps you can take to overcome this sense of loneliness.

Firstly, try to focus on self-care and nurturing your own well-being. Take some time each day to engage in activities that bring you joy

In [ ]:
import json
import faiss
import numpy as np

# Clear memory file
with open("/content/drive/MyDrive/user_memory.json", "w") as f:
    json.dump([], f)

# Create empty FAISS index
dimension = 384  # MiniLM embedding size

index = faiss.IndexFlatIP(dimension)

faiss.write_index(
    index,
    "/content/drive/MyDrive/memory_index.faiss"
)

print("Memory system reset.")

Memory system reset.


In [ ]:
memory_manager.load()
print(len(memory_manager.memories))

6


In [ ]:
def ember_chat(user_message):

    # 1. Safety Layer
    risk = detect_risk(
        user_message
    )

    # 2. Build RAG Context
    context = build_context(
        user_message
    )

    prompt = f"""
You are EMBER-X, a compassionate mental health support assistant.

Use the retrieved memories when they are relevant.

If a previous memory relates to the user's current concern, acknowledge it.

Do NOT generate USER responses.
Do NOT generate dialogue.
Only generate the therapist response.

{context}

Response:
"""

    # 3. Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    # 4. Generate Response
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # 5. Extract Therapist Response
    if "Therapist:" in response:
        response = response.split("Response:")[-1].strip()

    # 6. Crisis Escalation
    if risk == "high":

        response += "\n\n" + CRISIS_MESSAGE


    print("\n===== PROMPT =====\n")
    print(prompt[:3000])
    print("\n==================\n")


    # 7. Store Memory
    memory_manager.add_memory(user_message)

    return response

In [ ]:
print(
    ember_chat(
        "I feel lonely and hopeless."
    )
)

I'm here to offer you support and guidance as you navigate these difficult emotions. It's completely normal to feel lonely and disconnection at times, especially during challenging periods in life. Remember, it's okay to reach out to others for comfort and understanding. I encourage you to continue exploring various avenues for social connection, whether it's through support groups, community events, or simply opening up to trusted friends and loved ones. Taking small steps towards connecting with others can make a significant difference in your emotional wellbeing.

Additionally, it might be helpful to explore your thoughts and feelings surrounding loneliness and hopelessness. Reflecting on past experiences, triggers, and patterns can shed light on underlying beliefs and attitudes that contribute to these emotions. By gaining insight into these areas, you may discover ways to challenge negative thinking patterns and foster a more positive outlook.

Incorporating self-care practices in

In [ ]:
while True:

    user_input = input("\nYou: ")

    if user_input.lower() in [
        "exit",
        "quit"
    ]:

        print("\nEMBER-X: Goodbye!")
        break

    response = ember_chat(
        user_input
    )

    print(
        "\nEMBER-X:",
        response
    )


You: i am so tensed about my exams as it is coming near

EMBER-X: I understand how stressful it can be when exams are approaching, and it's completely normal to feel tense. Exam preparation can certainly bring up anxiety and apprehension. To help manage your stress and stay focused, here are a few strategies you might find useful:

1. Prioritize your study time: Break down your study materials into manageable sections and allocate specific times for each subject. This can help create a clear structure and prevent overwhelm.

2. Create a conducive environment: Find a quiet space where you can study without distractions. Turn off notifications on your phone and establish boundaries around study time to minimize interruptions.

3. Manage your time effectively: Use planners or digital tools to keep track of your assignments, deadlines, and exam dates. Set realistic goals and break them down into smaller, achievable tasks.

4. Practice relaxation techniques: Deep breathing exercises, progr

In [ ]:
while True:

    user_input = input("\nYou: ")

    if user_input.lower() in [
        "exit",
        "quit"
    ]:

        print("\nEMBER-X: Goodbye!")
        break

    response = ember_chat(
        user_input
    )

    print(
        "\nEMBER-X:",
        response
    )


You: i feel stressed again and back to that old mindset

===== PROMPT =====


You are EMBER-X, a compassionate mental health support assistant.

Use the retrieved memories when they are relevant.

If a previous memory relates to the user's current concern, acknowledge it.

Do NOT generate USER responses.
Do NOT generate dialogue.
Only generate the therapist response.


[USER MEMORY]

- I feel stressed again.
- I am stressed because of exams.
- I am stressed about exams.

[THERAPY EXAMPLES]

Example 1:

User:
Greetings, counselor. The weight of stress has become unbearable, impacting my overall well-being. My goal through this counseling session is to gain tools and strategies to better manage stress and find balance amidst life's demands.

Stress follows me relentlessly throughout the days, making it difficult to relax or find enjoyment in simple pleasures. Work pressures, financial obligations, and personal relationships all contribute to a constant sense of worry and unease. Racing 

In [ ]:
results = memory_manager.retrieve(
    "I feel stressed",
    top_k=5
)

for r in results:
    print(r["memory_summary"])

I feel stressed again.
I am stressed because of exams.
I am stressed about exams.
I feel lonely and hopeless.
I keep getting nightmares.


In [ ]:
memory_manager.add_memory(
    "My parents compare me with my cousin Rahul and it makes me feel worthless."
)

True

In [ ]:
results = memory_manager.retrieve(
    "My parents compared me again today",
    top_k=5
)

for m in results:
    print(m["memory_summary"])

My parents compare me with others.
You: My parents compare me with others and it makes me feel worthless.
My parents compare me with my cousin Rahul and it makes me feel worthless.
i feel stressed again and back to that old mindset
I feel stressed again.


In [ ]:
memory_manager.add_memory(
    "My dog Bruno died last year and I still miss him."
)

False